## Open notebook in:
| Colab
:---|
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nicolepcx/transformers-the-definitive-guide/blob/master/CH10/ch10_rLLM.ipynb)

# About This Notebook

The rLLM API changed after the book went to print. The chapter builds a coding
agent from `ToolAgent`, `ToolEnvironment` and `AgentExecutionEngine`, driven by
scripts in `rllm/examples/coder_tool/`. rLLM 0.3 removed all three classes and
deleted that directory. Agents are now async functions decorated with
`@rllm.rollout`, and evaluation runs through an `rllm eval` CLI.

| Book (0.1.x) | Now (0.3.x) |
|---|---|
| `ToolAgent` + `ToolEnvironment` | one `@rllm.rollout` function |
| `AgentExecutionEngine` | the runner behind `rllm eval` |
| `env_args["reward_fn"]` | a `@rllm.evaluator` function |
| `AgentTrainer(agent_class=…, env_class=…)` | `AgentTrainer(backend=…, agent_flow=…, evaluator=…)` |
| `prepare_code_data.py --train_size` | `prepare_deepcoder_data.py --train-size` |
| `run_code_with_tool.py` | `rllm eval` |
| `--limit` / `--parallel` / `--repeat` | `--max-examples` / `--concurrency` / `--attempts` |
| `compute_pass_at_k(results)` | reported automatically |

The upstream replacement, `cookbooks/deepcoder`, is single-turn — no Python
tool. So this notebook runs it as a baseline and adds a tool-using agent that
matches the chapter, built on the pattern from `cookbooks/math_tool_agent`.

Pinned to rLLM commit `75926c15e58f`.
<br>
<br>

<font color="red" size="6">
<b>ATT: There is a dependencies problem with Colab:</b>
</font>

<br>
<font color="black" size="4">
<b>To resolve this, go to: Runtime -> Change Runtime and then select:
Runtime version 2026.04</b>
</font>



# Install Dependencies

In [5]:
COMMIT = "75926c15e58fa29e4183d01292d10462d2047be9"

# Clean slate — re-running this cell from inside the repo is what nested
# rllm/rllm/rllm/... last time.
%cd /content
!pip uninstall -qy rllm coder-tool-agent
!rm -rf rllm

!git clone -q https://github.com/rllm-org/rllm.git
%cd rllm
!git checkout -q {COMMIT}
!git log -1 --oneline

/content
/content/rllm
75926c15 (HEAD, origin/main, origin/HEAD, main) feat(data): support pinned revisions in the SWE-Smith builder (#807)


In [6]:
# Core package.
!pip -q install -e .

# rLLM maps rllm-model-gateway to the in-repo copy via [tool.uv.sources], which
# is a uv-only table — pip ignores it and pulls an older build from PyPI (same
# 0.1.0 version string, missing create_app(local_handler=...)). Install the
# in-repo one explicitly to override it.
!pip -q install -e rllm-model-gateway

# The single-turn DeepCoder cookbook (our baseline). Cookbooks are plugins:
# they install separately and register with the CLI via entry points, so a
# plain `pip install git+...` of the core package is not enough.
!pip -q install --no-deps -e cookbooks/deepcoder

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.1/41.1 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.1/41.1 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 kB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.3/26.3 MB 132.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.8/140.8 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.0/132.0 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 847.1/847.1 kB 86.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.2/249.2 kB 39.3 MB/s e

In [2]:
!uv pip install vllm

Using Python 3.12.13 environment at: /usr
Resolved 194 packages in 1.71s
Prepared 93 packages in 29.33s
Uninstalled 16 packages in 3.17s
Installed 93 packages in 436ms
 + anthropic==0.120.2
 + apache-tvm-ffi==0.1.10
 + astor==0.8.1
 + blake3==1.0.9
 + cbor2==6.1.4
 + compressed-tensors==0.17.0
 - cuda-bindings==12.9.4
 + cuda-bindings==13.3.1
 - cuda-core==0.3.2
 + cuda-core==1.0.1
 - cuda-python==12.9.4
 + cuda-python==13.3.1
 + cuda-tile==1.5.0
 - cuda-toolkit==12.8.1
 + cuda-toolkit==13.0.2
 + depyf==0.20.0
 + detect-installer==0.1.0
 + dnspython==2.8.0
 + email-validator==2.3.0
 + fastapi-cli==0.0.32
 + fastapi-cloud-cli==0.23.0
 + fastar==0.11.0
 + fastsafetensors==0.3.3
 + flashinfer-python==0.6.14
 + humming-kernels==0.1.10
 + ijson==3.5.1
 + interegular==0.3.3
 + jmespath==1.1.0
 - lark==1.3.1
 + lark==1.2.2
 + llguidance==1.7.6
 - llvmlite==0.43.0
 + llvmlite==0.47.0
 + lm-format-enforcer==0.11.3
 + loguru==0.7.3
 + mistral-common==1.11.7
 + model-hosting-container-standards==

# Build the Tool-Using Coder Agent

In [7]:
!mkdir -p /content/rllm/cookbooks/coder_tool_agent

In [8]:
%%writefile /content/rllm/cookbooks/coder_tool_agent/coder_tool_agent.py
"""Multi-turn coding agent with a Python tool.

The AgentFlow equivalent of the pre-0.3 ToolAgent + ToolEnvironment recipe.
Loop: call the LLM with the python tool; if it calls the tool, run the code and
feed back stdout; if it replies without calling the tool, that message is the
submission.
"""

import asyncio
import json

from openai import AsyncOpenAI

import rllm
from rllm.tools.code_tools.python_interpreter import PythonInterpreter
from rllm.types import AgentConfig, Episode, Step, Task, Trajectory

MAX_TURNS = 8              # was env_args["max_steps"]
TOOL_TIMEOUT = 10          # was env_args["tool_time_limit_sec"]
MAX_TOOL_CHARS = 4000      # was env_args["max_tool_output_chars"]

SYSTEM_PROMPT = """\
You are a coding assistant that writes and executes Python functions to solve
programming problems.

Use the `python` tool to test your solution against the examples in the problem.
When you are satisfied, reply WITHOUT calling the tool and put the final
solution in a single fenced code block:

```python
# your solution here
```

Only that final tool-free message is graded.
"""

# backend="local" is what tools=["python"] used to resolve to. It runs code
# in-process with a timeout — fine for benchmark data, not a real sandbox.
interpreter = PythonInterpreter(backend="local")
TOOLS = [interpreter.json]


def to_dict(msg):
    """OpenAI message object -> plain dict."""
    if isinstance(msg, dict):
        return msg
    d = {"role": msg.role}
    if msg.content:
        d["content"] = msg.content
    if getattr(msg, "tool_calls", None):
        d["tool_calls"] = [
            {"id": tc.id, "type": tc.type,
             "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
            for tc in msg.tool_calls
        ]
    if getattr(msg, "tool_call_id", None):
        d["tool_call_id"] = msg.tool_call_id
    return d


@rllm.rollout(name="coder-tool-agent")
async def coder_tool_agent(task: Task, config: AgentConfig) -> Episode:
    client = AsyncOpenAI(base_url=config.base_url, api_key="EMPTY")
    question = task.metadata.get("question") or task.instruction

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    steps, answer, n_tool_calls = [], "", 0

    for _ in range(MAX_TURNS):
        response = await client.chat.completions.create(
            model=config.model, messages=messages, tools=TOOLS, timeout=600,
        )
        msg = response.choices[0].message
        messages.append(to_dict(msg))
        steps.append(Step(chat_completions=list(messages),
                          model_response=msg.content or "",
                          action=msg.content or ""))

        if not msg.tool_calls:
            answer = msg.content or ""      # no tool call -> this is the submission
            break

        for tc in msg.tool_calls:
            n_tool_calls += 1
            code = json.loads(tc.function.arguments)["code"]
            # The interpreter returns tracebacks as text, so errors come back
            # to the model as feedback rather than raising.
            out = await asyncio.to_thread(interpreter.forward, code=code, timeout=TOOL_TIMEOUT)
            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "content": out.to_string()[:MAX_TOOL_CHARS]})

    return Episode(
        trajectories=[Trajectory(name="coder", steps=steps)],
        artifacts={"answer": answer, "num_tool_calls": n_tool_calls},
    )

Writing /content/rllm/cookbooks/coder_tool_agent/coder_tool_agent.py


In [9]:
%%writefile /content/rllm/cookbooks/coder_tool_agent/coder_tool_eval.py
"""Grades the submitted code against the hidden tests.

Same grader as cookbooks/deepcoder, so scores are directly comparable between
the single-turn baseline and this tool-using agent. Also reports tool_calls,
which is what tells you whether RL taught the agent to use its tool or to skip it.
"""

import rllm
from rllm.eval.types import EvalOutput, Signal
from rllm.rewards.code_reward import RewardCodeFn
from rllm.rewards.reward_types import RewardConfig
from rllm.types import Episode, Task


@rllm.evaluator
def coder_tool_evaluator(task, episode: Episode) -> EvalOutput:
    # Eval passes a Task; training passes the raw row dict.
    task_info = task.metadata if isinstance(task, Task) else task

    out = RewardCodeFn(RewardConfig())(
        task_info=task_info, action=episode.artifacts["answer"]
    )

    return EvalOutput(
        reward=float(out.reward),
        is_correct=bool(out.is_correct),
        signals=[
            Signal(name="accuracy", value=1.0 if out.is_correct else 0.0),
            Signal(name="tool_calls", value=float(episode.artifacts["num_tool_calls"])),
        ],
    )

Writing /content/rllm/cookbooks/coder_tool_agent/coder_tool_eval.py


In [10]:
%%writefile /content/rllm/cookbooks/coder_tool_agent/pyproject.toml
[build-system]
requires = ["setuptools>=64"]
build-backend = "setuptools.build_meta"

[project]
name = "coder-tool-agent"
version = "0.1.0"
requires-python = ">=3.10"
dependencies = ["rllm", "openai"]

[tool.setuptools]
py-modules = ["coder_tool_agent", "coder_tool_eval"]

[project.entry-points."rllm.agents"]
coder-tool-agent = "coder_tool_agent:coder_tool_agent"

[project.entry-points."rllm.evaluators"]
coder-tool-agent = "coder_tool_eval:coder_tool_evaluator"

Writing /content/rllm/cookbooks/coder_tool_agent/pyproject.toml


In [11]:
%%capture
!pip install --no-deps -e /content/rllm/cookbooks/coder_tool_agent

In [12]:
!rllm agent list

╭──────────────────┬───────────────────┬───────────────────┬───────────────────╮
│ Name             │ Source            │ Module            │ Description       │
├──────────────────┼───────────────────┼───────────────────┼───────────────────┤
│ aider            │ built-in          │ rllm.harnesses.a… │ Run Paul          │
│                  │                   │                   │ Gauthier's aider  │
│                  │                   │                   │ CLI inside the    │
│                  │                   │                   │ sandbox.          │
│ bash             │ built-in          │ rllm.harnesses.b… │ Multi-turn ReAct  │
│                  │                   │                   │ bash loop inside  │
│                  │                   │                   │ a sandbox.        │
│ claude-code      │ built-in          │ rllm.harnesses.c… │ Run the Claude    │
│                  │                   │                   │ Code CLI inside   │
│                  │        

# Prepare the Dataset

In [13]:
!python /content/rllm/cookbooks/deepcoder/prepare_deepcoder_data.py --train-size 200 --test-size 50

README.md: 2.96kB [00:00, 9.09MB/s]
primeintellect/train-00000-of-00005.parq(…): 100% 471M/471M [00:08<00:00, 52.8MB/s]
primeintellect/train-00001-of-00005.parq(…): 100% 155M/155M [00:04<00:00, 33.2MB/s]
primeintellect/train-00002-of-00005.parq(…): 100% 109M/109M [00:02<00:00, 44.3MB/s]
primeintellect/train-00003-of-00005.parq(…): 100% 219M/219M [00:05<00:00, 40.0MB/s]
primeintellect/train-00004-of-00005.parq(…): 100% 205M/205M [00:05<00:00, 40.5MB/s]
Generating train split: 100% 16252/16252 [00:11<00:00, 1393.09 examples/s]
taco/train-00000-of-00004.parquet: 100% 208M/208M [00:08<00:00, 25.1MB/s]
taco/train-00001-of-00004.parquet: 100% 178M/178M [00:04<00:00, 38.2MB/s]
taco/train-00002-of-00004.parquet: 100% 189M/189M [00:06<00:00, 29.3MB/s]
taco/train-00003-of-00004.parquet: 100% 288M/288M [00:05<00:00, 52.6MB/s]
Generating train split: 100% 7436/7436 [00:07<00:00, 977.73 examples/s] 
lcbv5/train-00000-of-00011.parquet: 100% 53.5M/53.5M [00:02<00:00, 18.6MB/s]
lcbv5/train-00001-of-00

In [21]:
!pkill -f "vllm serve"

# Start vLLM Server in Background

In [22]:
import json
import os
import subprocess
import time
import urllib.error
import urllib.request
from pathlib import Path

VLLM_HOST = os.environ.get("VLLM_HOST", "127.0.0.1")
VLLM_PORT = int(os.environ.get("VLLM_PORT", "8000"))

MODEL_ID = os.environ.get("MODEL_ID", "Qwen/Qwen3-4B")
SERVED_MODEL_NAME = os.environ.get("SERVED_MODEL_NAME", MODEL_ID)

VLLM_LOG = Path(
    os.environ.get(
        "VLLM_LOG",
        str(Path.cwd() / "qwen3_ocr_vllm.log"),
    )
)

MODELS_URL = f"http://{VLLM_HOST}:{VLLM_PORT}/v1/models"


def vllm_ready() -> bool:
    try:
        with urllib.request.urlopen(MODELS_URL, timeout=5) as response:
            payload = json.loads(response.read().decode("utf-8"))
        return bool(payload.get("data"))
    except (
        urllib.error.URLError,
        TimeoutError,
        json.JSONDecodeError,
        OSError,
    ):
        return False


def tail_log(path: Path, lines: int = 160) -> str:
    if not path.exists():
        return ""
    return "\n".join(
        path.read_text(encoding="utf-8", errors="replace").splitlines()[-lines:]
    )


if vllm_ready():
    print(f"vLLM already running: {MODELS_URL}")
    VLLM_PID = None

else:
    cmd = [
        "vllm",
        "serve",
        MODEL_ID,
        "--trust-remote-code",
        "--served-model-name",
        SERVED_MODEL_NAME,
        "--host",
        "0.0.0.0",
        "--port",
        str(VLLM_PORT),
        "--limit-mm-per-prompt",
        '{"image": 1}',
        "--mm-processor-cache-gb",
        "0",
        "--no-enable-prefix-caching",
        "--dtype",
        "bfloat16",
        "--enable-auto-tool-choice",
        "--tool-call-parser",
        "hermes",
    ]

    env = os.environ.copy()

    # Prevent vLLM from trying the DeepGEMM FP8 path.
    env["VLLM_USE_DEEP_GEMM"] = "0"

    # Optional, but useful in notebook environments.
    env["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

    with open(VLLM_LOG, "a", encoding="utf-8") as log_fp:
        process = subprocess.Popen(
            cmd,
            stdout=log_fp,
            stderr=subprocess.STDOUT,
            start_new_session=True,
            env=env,
        )

    VLLM_PID = process.pid

    print(f"Launched vLLM server. PID: {VLLM_PID}")
    print("Command:", " ".join(cmd))
    print(f"Log: {VLLM_LOG}")

    timeout_seconds = int(os.environ.get("VLLM_STARTUP_TIMEOUT", "1800"))
    deadline = time.monotonic() + timeout_seconds

    while time.monotonic() < deadline:
        if vllm_ready():
            print(f"vLLM ready: {MODELS_URL}")
            break

        if process.poll() is not None:
            raise RuntimeError(
                "vLLM exited early.\n\n"
                f"Log tail:\n{tail_log(VLLM_LOG)}"
            )

        time.sleep(5)

    else:
        raise RuntimeError(
            "Timed out waiting for vLLM.\n\n"
            f"Log tail:\n{tail_log(VLLM_LOG)}"
        )

Launched vLLM server. PID: 48657
Command: vllm serve Qwen/Qwen3-4B --trust-remote-code --served-model-name Qwen/Qwen3-4B --host 0.0.0.0 --port 8000 --limit-mm-per-prompt {"image": 1} --mm-processor-cache-gb 0 --no-enable-prefix-caching --dtype bfloat16 --enable-auto-tool-choice --tool-call-parser hermes
Log: /content/rllm/qwen3_ocr_vllm.log
vLLM ready: http://127.0.0.1:8000/v1/models


# Evaluate

20 problems x 4 attempts = 80 trajectories, same shape as the book's run.
`--attempts` is the pass@k repeat count; `--concurrency` is throughput.

Single-turn baseline first:

In [14]:
!rllm eval deepcoder \
    --agent deepcoder \
    --evaluator deepcoder \
    --model Qwen/Qwen3-4B \
    --base-url http://localhost:8000/v1 \
    --split test \
    --max-examples 20 \
    --attempts 4 \
    --concurrency 32

  Tip: Try rllm UI for live monitoring! Run rllm login to get started.
  Materialised existing dataset on the fly at /root/.rllm/datasets/deepcoder
  Using materialised dataset at /root/.rllm/datasets/deepcoder

╭─────────────────────────────────────────────────────────────╮
│   Benchmark       deepcoder  (test, 20 examples)            │
│   Model           Qwen/Qwen3-4B                             │
│   Agent           deepcoder                                 │
│   Evaluator       deepcoder (overrides per-task verifier)   │
╰─────────────────────────────────────────────────────────────╯

Generating trajectories:   0% 0/80 [00:00<?, ?it/s][0:0] Rollout completed. Rewards: [deepcoder: 1.0] in 18s (setup=3s agentflow=14s [llm=14s/1 step] evaluator=1s teardown=0s), Termination: TerminationReason.ENV_DONE
Generating trajectories:   1% 1/80 [00:18<23:50, 18.11s/it][11:3] Rollout completed. Rewards: [deepcoder: 1.0] in 200s (setup=2s agentflow=197s [llm=195s/1 step] evaluator=1s teardown=0s

Then the tool-using agent, same flags:

In [23]:
!rllm eval deepcoder \
    --agent coder-tool-agent \
    --evaluator coder-tool-agent \
    --model Qwen/Qwen3-4B \
    --base-url http://localhost:8000/v1 \
    --split test \
    --max-examples 20 \
    --attempts 4 \
    --concurrency 8

  Tip: Try rllm UI for live monitoring! Run rllm login to get started.
  Using materialised dataset at /root/.rllm/datasets/deepcoder

╭────────────────────────────────────────────────────────────────────╮
│   Benchmark       deepcoder  (test, 20 examples)                   │
│   Model           Qwen/Qwen3-4B                                    │
│   Agent           coder-tool-agent                                 │
│   Evaluator       coder-tool-agent (overrides per-task verifier)   │
╰────────────────────────────────────────────────────────────────────╯

Generating trajectories:   0% 0/80 [00:00<?, ?it/s][0:0] Rollout completed. Rewards: [coder: 1.0] in 7s (setup=0s agentflow=6s [llm=5s/1 step] evaluator=0s teardown=0s), Termination: TerminationReason.ENV_DONE
Generating trajectories:   1% 1/80 [00:06<08:49,  6.71s/it][0:1] Rollout completed. Rewards: [coder: 1.0] in 9s (setup=0s agentflow=8s [llm=8s/1 step] evaluator=0s teardown=0s), Termination: TerminationReason.ENV_DONE
Generating

The tool run also reports `tool_calls`. If that trends to zero during RL the
agent learned to skip the tool, not to use it well.


In [24]:
!rllm view

  Browsing all runs under /root/.rllm/eval_results
  Serving rLLM episode viewer at http://127.0.0.1:7860/
  Root: /root/.rllm/eval_results
  Press Ctrl+C to stop.

  Shutting down…


# Train with GRPO

`AgentTrainer` now takes a flow and an evaluator instead of `agent_class` and
`env_class`. GRPO settings are Hydra overrides: `algorithm.adv_estimator=grpo`,
`actor_rollout_ref.rollout.n=4` for the group size.

In [25]:
!rllm train deepcoder \
    --agent coder-tool-agent \
    --evaluator coder-tool-agent \
    --model Qwen/Qwen3-4B-Instruct-2507 \
    --group-size 4 \
    --batch-size 16 \
    --lora-rank 32 \
    --epochs 1

  Tip: Try rllm UI for live monitoring! Run rllm login to get started.

╭───────────────────────────────── rLLM Train ─────────────────────────────────╮
│   Benchmark         deepcoder                                                │
│   Model             Qwen/Qwen3-4B-Instruct-2507                              │
│   Agent             coder-tool-agent                                         │
│   Evaluator         coder-tool-agent (overrides per-task verifier)           │
│   Train data        deepcoder  (train, 200 examples)                         │
│   Val data          deepcoder  (test, 50 examples)                           │
│   Sampling          train={'temperature': 1.0, 'top_p': 1.0, 'max_tokens':   │
│                     30720} (gateway-enforced)                                │
│   Group size        4                                                        │
│   Batch size        16                                                       │
│   Learning rate     2e-05          